# Storytelling Data Visualization on Exchange Rates

In this project, we build an explanatory, storytelling data visualization about the euro to US dollar exchange rate. The data comes from [Kaggle](https://www.kaggle.com/lsind18/euro-exchange-daily-rates-19992020) and was put together by Daria Chemkaeva using data from the European Central Bank. It contains the daily exchange rates of the euro against a number of currencies between 1999 and January 2021.

**The story we want to tell:** how the euro–dollar exchange rate evolved under the last three US presidents — George W. Bush (2001–2009), Barack Obama (2009–2017) and Donald Trump (2017–2021).

## Introducing the Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.style as style
%matplotlib inline

exchange_rates = pd.read_csv('euro-daily-hist_1999_2020.csv')
exchange_rates.head()

In [ ]:
exchange_rates.tail()

In [ ]:
exchange_rates.info()

The dataset has one row per business day and one column per currency. The date column is called `Period\\Unit:` and most currency columns are stored as strings (`object`). Several currencies have many null values because they weren't tracked for the whole period.

## Data Cleaning

In [ ]:
exchange_rates.rename(columns={'[US dollar ]': 'US_dollar',
                               'Period\\Unit:': 'Time'},
                      inplace=True)
exchange_rates['Time'] = pd.to_datetime(exchange_rates['Time'])
exchange_rates.sort_values('Time', inplace=True)
exchange_rates.reset_index(drop=True, inplace=True)

euro_to_dollar = exchange_rates[['Time', 'US_dollar']].copy()
euro_to_dollar['US_dollar'].value_counts()

In [ ]:
print((euro_to_dollar['US_dollar'] == '-').sum())

euro_to_dollar = euro_to_dollar[euro_to_dollar['US_dollar'] != '-'].copy()
euro_to_dollar['US_dollar'] = euro_to_dollar['US_dollar'].astype(float)
euro_to_dollar.info()

The `US_dollar` column contained **62** rows with the value `-` (days with no rate recorded), which also explains why the column was stored as strings. After removing them, we're left with **5,637** daily rates, and the column is converted to `float`.

## Rolling Mean

Daily exchange rates are noisy. To show the long-term trend, we plot the rolling mean over a 30-day window instead of the daily values.

In [ ]:
euro_to_dollar['rolling_mean'] = euro_to_dollar['US_dollar'].rolling(30).mean()

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(euro_to_dollar['Time'], euro_to_dollar['US_dollar'])
axes[0].set_title('Daily rate')
axes[1].plot(euro_to_dollar['Time'], euro_to_dollar['rolling_mean'])
axes[1].set_title('30-day rolling mean')
plt.show()

## Coming Up with an Idea

We'll show how the euro–dollar rate changed under the last three US presidents. Splitting the timeline by presidency gives the reader three natural "chapters", and the change within each chapter makes it a data story rather than a simple chart.

## Sketching the Graph

The sketch we followed:

- A title that states the main finding, and a subtitle explaining what is shown.
- A single line chart of the 30-day rolling mean from 2001 to 2021, with each presidency highlighted in its own color and the rest of the line in grey.
- A label with each president's name and years above their period, plus the average rate during that period.
- A FiveThirtyEight-style layout with minimal ticks and a signature bar at the bottom.

## Coding the Graph

In [ ]:
bush = euro_to_dollar[(euro_to_dollar['Time'] >= '2001-01-20') & (euro_to_dollar['Time'] < '2009-01-20')]
obama = euro_to_dollar[(euro_to_dollar['Time'] >= '2009-01-20') & (euro_to_dollar['Time'] < '2017-01-20')]
trump = euro_to_dollar[(euro_to_dollar['Time'] >= '2017-01-20') & (euro_to_dollar['Time'] < '2021-01-20')]
bush_obama_trump = euro_to_dollar[euro_to_dollar['Time'] >= '2001-01-01']

for name, period in [('Bush', bush), ('Obama', obama), ('Trump', trump)]:
    start = period['rolling_mean'].iloc[0]
    end = period['rolling_mean'].iloc[-1]
    print(name, round(period['US_dollar'].mean(), 3),
          round(period['US_dollar'].min(), 3), round(period['US_dollar'].max(), 3),
          round(start, 3), round(end, 3))

peak = euro_to_dollar.loc[euro_to_dollar['US_dollar'].idxmax()]
print(peak['Time'].date(), peak['US_dollar'])

In [ ]:
style.use('fivethirtyeight')

colors = {'Bush': '#BF5FFF', 'Obama': '#ffa500', 'Trump': '#00B2EE'}

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(bush_obama_trump['Time'], bush_obama_trump['rolling_mean'], color='#A6D785', alpha=0.3, linewidth=2)
for name, period in [('Bush', bush), ('Obama', obama), ('Trump', trump)]:
    ax.plot(period['Time'], period['rolling_mean'], color=colors[name], linewidth=2.5)

ax.set_ylim(0.8, 1.7)
ax.set_yticks([0.8, 1.0, 1.2, 1.4, 1.6])
ax.set_yticklabels(['0.8', '1.0', '1.2', '1.4', '1.6'], alpha=0.4)
ax.tick_params(axis='x', colors='grey')
ax.grid(alpha=0.5)

labels = [('BUSH', '(2001-2009)', bush), ('OBAMA', '(2009-2017)', obama), ('TRUMP', '(2017-2021)', trump)]
for name, years, period in labels:
    mid = period['Time'].iloc[len(period) // 2]
    ax.text(mid, 1.62, name, color=colors[name.title()], weight='bold', ha='center', fontsize=14)
    ax.text(mid, 1.56, years, color=colors[name.title()], ha='center', fontsize=11)
    ax.text(mid, 1.50, 'avg: {:.2f}'.format(period['US_dollar'].mean()), color='grey', ha='center', fontsize=11)

ax.text(pd.Timestamp('1999-06-01'), 1.86, 'EURO-USD rate peaked in 2008, then fell under Obama and stayed low under Trump',
        weight='bold', fontsize=15)
ax.text(pd.Timestamp('1999-06-01'), 1.78, 'EURO-USD exchange rates (30-day rolling mean) under George W. Bush, Barack Obama, and Donald Trump',
        fontsize=12)

ax.text(pd.Timestamp('1999-06-01'), 0.65,
        '©DATAQUEST' + ' ' * 140 + 'Source: European Central Bank',
        color='#f0f0f0', backgroundcolor='#4d4d4d', size=11)

plt.show()

The graph tells a three-chapter story:

- **Bush (2001–2009):** the euro strengthened steadily against the dollar — the 30-day average went from about 0.92 to 1.35, and the rate reached its all-time high in this dataset, **1.599**, on July 15, 2008, just before the financial crisis. Average rate: **1.20**.
- **Obama (2009–2017):** the rate was volatile and highest on average (**1.28**), but it trended down, from about 1.35 at the start to about 1.05 at the end of the term.
- **Trump (2017–2021):** the rate stayed in a narrower and lower band (between 1.05 and 1.25), with the lowest average of the three periods, **1.14**, before rising again at the end of 2020.

We used color to separate the three presidencies (similarity), labels placed directly above each period (proximity) instead of a legend, a title that states the main finding, and a FiveThirtyEight-style layout to keep the focus on the data.

## Next Steps

In this project, we cleaned the euro–dollar exchange rate data, smoothed it with a 30-day rolling mean, and built a storytelling visualization showing how the rate evolved under three US presidents.

Possible next steps:

- Mark key events on the chart (the 2008 financial crisis, the 2010–2012 euro debt crisis, the 2020 COVID-19 pandemic) to explain the biggest moves.
- Compare the euro against other currencies (e.g., the British pound or the Chinese yuan) over the same periods.
- Show the percentage change within each presidency instead of the absolute rate.